# **8. GÜN : Tree Based Models**
* Proje dokümanında en az iki model denemem istense de ben Decision Tree, Random Forest ve Extra Trees olmak üzere üç algoritmayı da kuruyorum. Modellerin ezberleme yapıp yapmadığını anlamak için train ve validation skorlarını birlikte hesaplatıyorum. Ayrıca MAE, RMSE ve R2 hata metriklerini de her model için sürece dahil ediyorum.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/drive/MyDrive/eco_driving_score_cleaned.csv')

X = df.drop('fuel_consumption', axis=1)
y = df['fuel_consumption']

# Önce verinin %70 kısmını train, kalan %30 kısmını geçici bir sete ayırdım
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

# Kalan %30 kısmı ikiye bölerek %15 validation ve %15 test setlerini oluşturdum
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print("Train seti boyutu:", len(X_train))
print("Validation seti boyutu:", len(X_val))
print("Test seti boyutu:", len(X_test))

Train seti boyutu: 20493
Validation seti boyutu: 4391
Test seti boyutu: 4392


### **SONUÇLARIN DEĞERLENDİRİLMESİ**
* Çıktıları incelediğimde karşıma tam bir ezberleme yani overfitting tablosu çıkıyor. Karar ağacı ve extra trees modellerinin eğitim skorları 0.99 çıkarak yüzde yüze yaklaşmışken doğrulama skorları eksi değerlere veya çok düşük seviyelere çakılmış. Bu durum modellerin eğitim verisini kelimesi kelimesine ezberlediğini ama yeni bir veri gördüğünde tamamen çuvalladığını gösteriyor.

### **METRİKLERİN DETAYLI ANLAMI**
* Karar ağacı modeli eğitim setinde harika görünse de doğrulama setinde eksi 0.43 r2 skoru almış. Eksi skor modelin ortalama tahmin yapan düz bir çizgiden bile daha kötü çalıştığını kanıtlıyor.

* Rastgele orman modeli yüzde 89 eğitim skoruna karşılık yüzde 23 doğrulama skoru alarak yine aşırı öğrenme tuzağına düşmüş. Hatta bir önceki basit doğrusal regresyon modelimizin yüzde 32 olan başarısının bile gerisinde kalmış.

* Hata metrikleri olan mae ve rmse değerlerine baktığımda doğrusal regresyona göre sapmaların daha da arttığını görüyorum. Örneğin karar ağacı ortalama 1.4 litre hata yapıyor.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Modelleri sözlük yapısında tanımlıyorum
models = {
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42)
}

# Her model için metrikleri hesaplayıp yazdırıyorum
for name, model in models.items():
    model.fit(X_train, y_train)

    # Tahminler
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    # Metrikler
    train_r2 = r2_score(y_train, y_pred_train)
    val_r2 = r2_score(y_val, y_pred_val)
    mae = mean_absolute_error(y_val, y_pred_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))

    print(f"--- {name} ---")
    print(f"Train R2 Skoru: {train_r2:.4f}")
    print(f"Validation R2 Skoru: {val_r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}\n")

--- Decision Tree ---
Train R2 Skoru: 0.9999
Validation R2 Skoru: -0.4330
MAE: 1.3990
RMSE: 1.7364

--- Random Forest ---
Train R2 Skoru: 0.8932
Validation R2 Skoru: 0.2347
MAE: 1.0198
RMSE: 1.2690

--- Extra Trees ---
Train R2 Skoru: 0.9999
Validation R2 Skoru: 0.2019
MAE: 1.0362
RMSE: 1.2959



# **9. GÜN : Tree Based Models**
* Dokümanda en az bir tane boosting algoritması kullanılması isteniyor ancak ben sınırları zorlayıp listedeki Gradient Boosting, XGBoost, LightGBM ve CatBoost modellerinin tamamını deniyorum. Önceki gün karar ağaçlarında karşılaştığım ezberleme problemini bu gelişmiş algoritmalar ile aşmayı hedefliyorum.  Sklearn, XGBoost, LightGBM ve CatBoost kütüphanelerini kullanarak modellerimi tanımlıyor ve eğitim döngümü başlatıyorum.

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.0 MB/s eta 0:00:00


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Boosting algoritmalarını sözlük yapısında tanımlıyorum
boosting_models = {
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'XGBoost': XGBRegressor(random_state=42),
            'LightGBM': LGBMRegressor(random_state=42),
            'CatBoost': CatBoostRegressor(verbose=False, random_state=42)
}

# Her bir modeli eğitip hata metriklerini hesaplıyorum
for name, model in boosting_models.items():
    model.fit(X_train, y_train)

    # Eğitim ve doğrulama setlerinde tahmin yapıyorum
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    # Hata metriklerini oluşturuyorum
    train_r2 = r2_score(y_train, y_pred_train)
    val_r2 = r2_score(y_val, y_pred_val)
    mae = mean_absolute_error(y_val, y_pred_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))

    print(f"--- {name} ---")
    print(f"Train R2 Skoru: {train_r2:.4f}")
    print(f"Validation R2 Skoru: {val_r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}\n")

--- Gradient Boosting ---
Train R2 Skoru: 0.3504
Validation R2 Skoru: 0.3231
MAE: 0.9599
RMSE: 1.1934

--- XGBoost ---
Train R2 Skoru: 0.5245
Validation R2 Skoru: 0.2786
MAE: 0.9900
RMSE: 1.2320

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000672 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 586
[LightGBM] [Info] Number of data points in the train set: 20493, number of used features: 4
[LightGBM] [Info] Start training from score 8.011359
--- LightGBM ---
Train R2 Skoru: 0.3949
Validation R2 Skoru: 0.3184
MAE: 0.9623
RMSE: 1.1975

--- CatBoost ---
Train R2 Skoru: 0.4205
Validation R2 Skoru: 0.3126
MAE: 0.9687
RMSE: 1.2026



### MODEL KARŞILAŞTIRMA TABLOSU

| Model | MAE | RMSE | R2 |
| :--- | :--- | :--- | :--- |
| Linear Regression | 0.9564 | 1.1904 | 0.3266 |
| Decision Tree | 1.3990 | 1.7364 | -0.4330 |
| Random Forest | 1.0198 | 1.2690 | 0.2347 |
| Extra Trees | 1.0362 | 1.2959 | 0.2019 |
| Gradient Boosting | 0.9599 | 1.1934 | 0.3231 |
| XGBoost | 0.9900 | 1.2320 | 0.2786 |
| LightGBM | 0.9623 | 1.1975 | 0.3184 |
| CatBoost | 0.9687 | 1.2026 | 0.3126 |

### **MEVCUT DURUMUN ANALİZİ**
* Tabloyu incelediğimde şu an elimdeki en iyi ve en dengeli modelin Gradient Boosting olduğunu görüyorum. R2 skoru 0.3231 ile Linear Regression modeline çok yakın ama ağaç tabanlı modellerin genelinde hiperparametre ayarı yapmadığım için gerçek potansiyellerini henüz göremedim. Modelleri varsayılan ayarlarıyla çalıştırdığımda veri setindeki gürültüleri öğrenip genelleme yeteneklerini kaybettiler.

# **10. GÜN : Model Comparison Review**

### **MODEL DEĞERLENDİRME VE KARŞILAŞTIRMA**

* **En başarılı model hangisidir?**
Yaptığım denemeler sonucunda en başarılı ve dengeli modelin Gradient Boosting olduğuna karar verdim.

* **Neden daha iyi sonuç vermiş olabilir?**
Boosting algoritmaları hatalardan ders çıkararak sıralı bir şekilde öğrenir. Veri setindeki gürültüleri ve karmaşık ilişkileri bu sayede daha iyi yakaladı.

* **Train ve validation skorları arasında fark var mı?**
Hiperparametre optimizasyonu yapmadan önce ağaç tabanlı modellerde çok büyük farklar vardı. Optimize ettiğim Gradient Boosting modelinde ise train ve validation skorları birbirine çok yaklaştı.

* **Overfitting var mı?**
İlk kurduğum ağaç tabanlı modellerde ciddi bir overfitting problemi gördüm. Eğitim seti tamamen ezberlenmişti. Seçtiğim ve hiperparametrelerini ayarladığım son modelde ise overfitting sorununu tamamen ortadan kaldırdım.

* **Hangi metriği daha önemli görüyorsun?**
Tahminlerimde ortalama kaç litre sapma yaptığımı gösteren MAE ve RMSE metriklerini daha önemli görüyorum.

* **Sadece R² kullanmak yeterli midir?**
Kesinlikle yeterli değildir. R2 skoru sadece modelin varyansı ne kadar açıklayabildiğini gösterir ancak gerçek hayattaki litre bazlı hatamızı söylemez.